In [1]:
import numpy as np
import sounddevice as sd
import threading

In [3]:
# Shared state
global_parameters = {"freq": 440.0, "amp": 0.5}
lock = threading.Lock()

In [4]:
samplerate=44000

In [5]:
def callback(outdata, frames, time, status):
    # print(outdata)
    with lock:
        freq = global_parameters["freq"]
        amp = global_parameters["amp"]
    # Generate audio using the copied parameters (outside the lock)
    t = np.linspace(0, frames / samplerate, frames, endpoint=False)
    outdata[:, 0] = amp * np.sin(2 * np.pi * freq * t) # Mono: left channel
    outdata[:, 1] = outdata[:, 0] * 0.8 # Stereo: right 

In [6]:
# Start stream
stream = sd.OutputStream(
    samplerate=samplerate,
    channels=2,
    callback=callback,
    dtype="float32"
)
stream.start()

In [13]:
# In another cell, update parameters safely:
with lock:
    global_parameters.update({"freq": 880.0, "amp": 0.3})

In [14]:
stream.stop()

In [15]:
stream.close()

In [2]:
import dissipative_sonification.config as cfg
from dissipative_sonification.sonification import AudioEngine

In [3]:
engine = AudioEngine()

In [4]:
engine.start()

In [8]:
state = {
            "f": 0.3,
            "k": 680.0,
            "v_mean": 0.0,
            "v_row": np.zeros(cfg.WIDTH),
            "entropy": 0.0,
            "gradient": 0.0,
            "pan": 0.0,
        }
engine.update_state(state)

In [9]:
engine.stop()